In [1]:
from absl import app

In [11]:
import os

model_path = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/saved_models/1D/modes16_width64_epochs500/burgers_1d_FNO_model_trainedby_dim1d_nx1024_N1500_solver=exponax_kernel=gaussian_correlation_length0.03_bcperiodic_nu0.0005_t1.0_seed45.pth"
print(f"Model exists: {os.path.exists(model_path)}")
print(f"File size: {os.path.getsize(model_path)} bytes" if os.path.exists(model_path) else 'File missing')

Model exists: True
File size: 2352752 bytes


In [12]:
try:
    with open(model_path, 'rb') as f:
        header = f.read(4)  # First 4 bytes should be PK\x03\x04 for zip files
        print(f"File header: {header}")
except Exception as e:
    print(f"File read error: {str(e)}")

File header: b'PK\x03\x04'


In [13]:
import torch
import os
import sys
project_root = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO"
sys.path.append(project_root)
from models.FNO1d import FNO1d

model = FNO1d(modes=16, width=64) 
model.load_state_dict(torch.load(model_path))
model.eval()

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [6]:
try:
    # Method 1: Standard load
    state_dict = torch.load(model_path, map_location='cpu')
    model.load_state_dict(state_dict)
    print("Successfully loaded via state_dict")
except Exception as e:
    print(f"Standard load failed: {str(e)}")
    
    try:
        # Method 2: Load entire model
        model = torch.load(model_path, map_location='cpu')
        print("Successfully loaded full model")
    except Exception as e:
        print(f"Full model load failed: {str(e)}")
        
        try:
            # Method 3: Try different torch version
            from torch.serialization import _legacy_load
            with open(model_path, 'rb') as f:
                model = _legacy_load(f, map_location='cpu')
            print("Successfully loaded with legacy method")
        except Exception as e:
            print(f"Legacy load failed: {str(e)}")
            raise RuntimeError("All loading methods failed")

Standard load failed: PytorchStreamReader failed reading zip archive: failed finding central directory
Full model load failed: PytorchStreamReader failed reading zip archive: failed finding central directory
Legacy load failed: _legacy_load() missing 1 required positional argument: 'pickle_module'


RuntimeError: All loading methods failed

In [9]:
import torch
from pathlib import Path
import os
import numpy as np

# Define paths
current_file_path = Path("/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO")
dataset_name = "dim1d_nx1024_N1500_solver=exponax_kernel=gaussian_correlation_length0.03_bcperiodic_nu0.0005_t1.0_seed45.pt"
file_path = current_file_path / "datasets" / "1D" / "Burgers" / dataset_name
data = torch.load(file_path)

RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

In [10]:
if os.path.isfile(file_path):
    print("File size (bytes):", os.path.getsize(file_path))
    with open(file_path, "rb") as f:
        header = f.read(4)
        print("File header:", header)

File size (bytes): 524288
File header: b'PK\x03\x04'


In [14]:
import jax
print(jax.devices())

[CpuDevice(id=0)]
